In [ ]:
# gt_pipeline_scaled.py
# Full Plan B pipeline scaled over all primitive terms.
# Each term gets its own folder under BASE_DIR.
# A diagnostic .txt is saved per term summarising bad months.

import os
import time
import random, math
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from pytrends.request import TrendReq

# ── Config ────────────────────────────────────────────────────────────────────
BASE_DIR   = Path("C:/Python/CSUREMM/dictionary V0.0/data_primitive")
START_DATE = pd.Timestamp("2022-01-01")
END_DATE   = pd.Timestamp("2026-05-31")
CHUNK_DAYS = 269
STEP_DAYS  = 240
SLEEP_SEC  = 8
GEO        = "US"
GPROP      = ""   # "" = web search only

PRIMITIVES = [
    "ABUNDANCE","ACCRUE","ADVANTAGE","AFFLUENCE","AFFLUENT","AFLOAT","ALLOWANCE",
    "ARISTOCRACY","ARISTOCRAT","ARISTOCRATIC","ASSOCIATE","BACKER","BACKWARD",
    "BACKWARDNESS","BANKRUPT","BANKRUPTCY","BARGAIN","BEGGAR","BENEFACTOR",
    "BENEFICIARY","BENEFIT","BENEVOLENCE","BENEVOLENT","BEQUEATH","BETROTH",
    "BETROTHAL","BLACKMAIL","BONUS","BOOM","BREADWINNER","BRIBE","BROKE","BUM",
    "BUY","CAPITALIZE","CHARITABLE","CHARITY","CHEAP","COLONY","COMMONER",
    "COMMUNITY","COMPENSATE","COMPENSATION","CONTRIBUTE","CONTRIBUTION",
    "COOPERATIVE","CORRUPT","COST","COSTLINESS","COSTLY","CRISIS","DEBTOR",
    "DEFAULT","DEFICIT","DEPRECIATION","DEPRESSION","DESTITUTE","DOMINATION",
    "DONATE","DONATION","ECONOMIZE","ENDOW","ENTREPRENEURIAL","EQUITY","EXPENSE",
    "EXPENSIVE","EXTRAVAGANT","FELLOWSHIP","FINE","FIRE","FRUGAL","GAIN","GAMBLE",
    "GENEROSITY","GHETTO","GIFT","GOLD","GUIDE","HOLE","HUSTLE","HUSTLER",
    "INEXPENSIVE","INFLATION","INHERIT","INTERVENTION","INVALUABLE","JOBLESS",
    "LAID","LAY","LEGAL","LIQUIDATE","LIQUIDATION","LUCRATIVE","LUXURY",
    "MERITORIOUS","MISER","NEGOTIATE","NOBILITY","NOBLEMAN","OWE","PARTNER",
    "PARTNERSHIP","PATRON","PATRONAGE","POLLUTION","POOR","POVERTY","PRECIOUS",
    "PRICELESS","PRIVILEGED","PRODUCTIVE","PRODUCTIVITY","PROFIT","PROFITABLE",
    "PROSPER","PROSPERITY","PROSPEROUS","RACE","RADICAL","RECESSION","RECOMPENSE",
    "REDEMPTION","REPARATION","REWARD","RICH","RICHES","RICHNESS","RUIN","SAVINGS",
    "SECURITY","SEGREGATION","SHORTAGE","SKILL","SQUANDER","STEAL","SUBSIDIZE",
    "SUBSIDY","SUCCESS","SUCCESSFUL","TARIFF","THRIFT","THRIFTY","TREASURE", 
    "UNDERWORLD","UNECONOMICAL", "UNEMPLOYED","UNPROFITABLE","VAGABOND","VAGRANT",
    "VALUABLE","WARFARE","WASTE","WORTH",]

# ── Helpers ───────────────────────────────────────────────────────────────────

def make_pytrends():
    return TrendReq(hl="en-US", tz=0, timeout=(10, 30), retries=3, backoff_factor=0.5)

def term_dir(term: str) -> Path:
    """Folder named after the lower-case term."""
    d = BASE_DIR / term.lower()
    d.mkdir(parents=True, exist_ok=True)
    return d

def window_filename(start, end, prefix):
    return f"{prefix}_{start.date()}_{end.date()}.csv"

def human_sleep(attempt: int, base: int = SLEEP_SEC):
    """
    Exponential backoff with full jitter.
    Attempt 1: 15–30s,  2: 15–60s,  3: 15–120s,  4: 15–240s,  5: 15–480s
    The random jitter means no two retries look the same to Google's detector.
    """
    cap   = base * (2 ** attempt)          # exponential ceiling: 30, 60, 120, 240, 480
    wait  = random.uniform(base, cap)      # random draw between floor and ceiling
    print(f"      sleeping {wait:.1f}s …")
    time.sleep(wait)

In [2]:
# ── Step 1: Download daily chunks ─────────────────────────────────────────────

def download_daily_chunks(term: str, folder: Path) -> list[str]:
    """
    Returns list of failed timeframes. Empty list = all chunks downloaded cleanly.
    """
    pytrends    = make_pytrends()
    chunk_start = START_DATE
    failed      = []

    while chunk_start <= END_DATE:
        chunk_end = min(chunk_start + pd.Timedelta(days=CHUNK_DAYS - 1), END_DATE)
        fname     = folder / window_filename(chunk_start, chunk_end, "gt_daily")

        if fname.exists():
            print(f"    [skip] {chunk_start.date()} → {chunk_end.date()} (already on disk)")
            chunk_start += pd.Timedelta(days=STEP_DAYS)
            continue

        timeframe = f"{chunk_start.date()} {chunk_end.date()}"
        saved     = False

        for attempt in range(1, 4):
            print(f"    [pull] {timeframe}  attempt {attempt}/3 …", end=" ", flush=True)
            try:
                pytrends.build_payload([term], cat=0, timeframe=timeframe, geo=GEO, gprop=GPROP)
                df = pytrends.interest_over_time()
                if df.empty or term not in df.columns:
                    print("empty — skipped")
                    break
                out = (df[[term]].reset_index()
                    .rename(columns={"date": "Time", term: "value"})[["Time", "value"]])
                out.to_csv(fname, index=False)
                print(f"{len(out)} rows → {fname.name}")
                saved = True
                break
            except Exception as e:
                if "429" in str(e):
                    print(f"429 —", end=" ")
                    human_sleep(attempt)
                    pytrends = make_pytrends()
                else:
                    print(f"ERROR: {e} — skipped")
                    break

        if not saved:
            print(f"    FAILED after 3 attempts: {timeframe}")
            failed.append(timeframe)

        chunk_start += pd.Timedelta(days=STEP_DAYS)
        time.sleep(random.uniform(8, 20))          # inter-chunk pause

    return failed                             # caller decides what to do


# ── Step 2: Download monthly anchor ───────────────────────────────────────────

def download_monthly_anchor(term: str, folder: Path) -> bool:
    """Returns True if anchor is available (existing or freshly downloaded)."""
    fname = folder / window_filename(START_DATE, END_DATE, "gt_monthly")
    if fname.exists():
        print(f"    [skip] monthly anchor (already on disk)")
        return True

    timeframe = f"{START_DATE.date()} {END_DATE.date()}"
    for attempt in range(1, 4):
        print(f"    [pull] monthly anchor {timeframe}  attempt {attempt}/3 …", end=" ", flush=True)
        try:
            pytrends = make_pytrends()
            pytrends.build_payload([term], cat=0, timeframe=timeframe, geo=GEO, gprop=GPROP)
            df = pytrends.interest_over_time()
            if df.empty or term not in df.columns:
                print("empty — not saved")
                return False
            out = (df[[term]].reset_index()
                   .rename(columns={"date": "Time", term: "monthly_gt"})[["Time", "monthly_gt"]])
            out.to_csv(fname, index=False)
            print(f"{len(out)} rows → {fname.name}")
            return True
        except Exception as e:
            if "429" in str(e):
                print(f"429 — waiting before retry …")
                time.sleep(random.uniform(8, 20)) 
            else:
                print(f"ERROR: {e} — skipped")
                return False

    print(f"    FAILED after 3 attempts: monthly anchor for '{term}'")
    return False




In [3]:
# ── Step 3: Classify files ─────────────────────────────────────────────────────

def classify_files(folder: Path):
    all_files    = list(folder.glob("*.csv"))
    daily_files  = []
    monthly_file = None

    for f in all_files:
        df         = pd.read_csv(f)
        dates      = pd.to_datetime(df["Time"])
        median_gap = dates.diff().dt.days.median()
        if median_gap <= 2:
            daily_files.append(f)
        else:
            monthly_file = f

    daily_files = sorted(daily_files, key=lambda f: pd.read_csv(f)["Time"].min())
    return daily_files, monthly_file



In [4]:
# ── Step 4: Stitch daily chunks ───────────────────────────────────────────────

def stitch_daily(daily_files: list) -> pd.DataFrame:
    dfs = []
    for f in daily_files:
        df = pd.read_csv(f, parse_dates=["Time"])
        df = df.rename(columns={c: "value" for c in df.columns if c != "Time"})
        df = df.sort_values("Time").drop_duplicates("Time")
        dfs.append(df)

    if not dfs:
        raise ValueError("No daily files to stitch.")

    stitched = dfs[0].copy()

    for i in range(1, len(dfs)):
        prev    = stitched
        curr    = dfs[i].copy()
        overlap = prev.merge(curr, on="Time", suffixes=("_prev", "_curr"))

        if len(overlap) >= 5:
            median_prev = overlap["value_prev"].median()
            median_curr = overlap["value_curr"].median()
            scale       = median_prev / median_curr if median_curr > 1e-9 else 1.0
        else:
            print(f"    WARNING: only {len(overlap)} overlap points for chunk {i} — scale=1.0")
            scale = 1.0

        curr         = curr.copy()
        curr["value"] *= scale
        combined     = pd.concat([prev, curr]).groupby("Time", as_index=False)["value"].mean()
        stitched     = combined.sort_values("Time").reset_index(drop=True)

    return stitched



In [5]:
# ── Step 5: Monthly anchor calibration (local per-month scale) ────────────────

def apply_monthly_anchor_local(stitched: pd.DataFrame, monthly_file) -> pd.DataFrame:
    monthly = pd.read_csv(monthly_file, parse_dates=["Time"]).set_index("Time").squeeze()
    monthly.name = "monthly"

    daily_agg = (stitched.set_index("Time")["value"]
                 .resample("MS").mean())
    daily_agg.name = "daily_agg"

    ratio = pd.concat([monthly, daily_agg], axis=1, sort=False).dropna()
    ratio = ratio[ratio["daily_agg"] > 1e-9]
    ratio["scale"] = ratio["monthly"] / ratio["daily_agg"]

    daily_scale = (ratio["scale"]
                   .reindex(pd.date_range(ratio.index.min(), ratio.index.max(), freq="D"))
                   .interpolate(method="time")
                   .ffill()
                   .bfill())

    result          = stitched.copy().set_index("Time")
    result["value"] = (result["value"] * daily_scale).clip(0, 100)
    return result.reset_index()


# ── Step 6: Save final series ─────────────────────────────────────────────────

def save_final(stitched: pd.DataFrame, term: str, folder: Path) -> pd.DataFrame:
    out   = stitched.rename(columns={"value": term.lower()})
    fname = folder / f"gt_stitched_{START_DATE.date()}_{END_DATE.date()}.csv"
    out.to_csv(fname, index=False)
    print(f"    Saved: {fname.name}  ({len(out)} rows)")
    return out


# ── Diagnostic txt ────────────────────────────────────────────────────────────

def save_diagnostic(stitched: pd.DataFrame, monthly_file, term: str, folder: Path):
    """
    Computes month-by-month ratio and writes a plain-text diagnostic report.
    Flags months outside [0.85, 1.15].
    """
    monthly   = pd.read_csv(monthly_file, parse_dates=["Time"]).set_index("Time").squeeze()
    daily_agg = (stitched.set_index("Time").iloc[:, 0]
                 .resample("MS").mean())
    daily_agg.name = "daily_agg"

    ratio = pd.concat([monthly, daily_agg], axis=1, sort=False).dropna()
    ratio.columns      = ["monthly", "daily_agg"]
    ratio["ratio"]     = ratio["daily_agg"] / ratio["monthly"].replace(0, pd.NA)
    bad                = ratio[ratio["ratio"].lt(0.85) | ratio["ratio"].gt(1.15)]

    lines = [
        f"Google Trends Diagnostic — term: {term.lower()}",
        f"Period : {START_DATE.date()} → {END_DATE.date()}",
        f"Rows in stitched series : {len(stitched)}",
        f"Monthly anchor points   : {len(ratio)}",
        f"",
        f"Months with ratio outside [0.85, 1.15]  ({len(bad)} found):",
        "",
    ]

    if bad.empty:
        lines.append("  None — alignment is clean.")
    else:
        lines.append(f"{'Time':<15} {'monthly':>10} {'daily_agg':>12} {'ratio':>8}")
        lines.append("-" * 50)
        for idx, row in bad.iterrows():
            lines.append(
                f"{str(idx.date()):<15} {row['monthly']:>10.1f} "
                f"{row['daily_agg']:>12.3f} {row['ratio']:>8.3f}"
            )

    lines += [
        "",
        "Full monthly comparison:",
        "",
        f"{'Time':<15} {'monthly':>10} {'daily_agg':>12} {'ratio':>8}",
        "-" * 50,
    ]
    for idx, row in ratio.iterrows():
        flag = "  ← !" if (row["ratio"] < 0.85 or row["ratio"] > 1.15) else ""
        lines.append(
            f"{str(idx.date()):<15} {row['monthly']:>10.1f} "
            f"{row['daily_agg']:>12.3f} {row['ratio']:>8.3f}{flag}"
        )

    txt_path = folder / "diagnostic.txt"
    txt_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"    Diagnostic saved: {txt_path.name}  ({len(bad)} bad months)")


In [7]:
# ── Main loop ─────────────────────────────────────────────────────────────────
START_FROM = "vagrant"   # set to the term where 429 started, or None to run all

if __name__ == "__main__":
    total  = len(PRIMITIVES)
    errors = []

    # Skip terms before START_FROM
    primitives_to_run = PRIMITIVES
    if START_FROM:
        start_idx = next(
            (i for i, t in enumerate(PRIMITIVES) if t.lower() == START_FROM.lower()),
            0
        )
        primitives_to_run = PRIMITIVES[start_idx:]
        print(f"Resuming from '{START_FROM}' ({len(primitives_to_run)} terms remaining)")
    else:
        start_idx = 0

    try:                                             # ← outer try wraps the for loop
        for i, raw_term in enumerate(primitives_to_run, start_idx + 1):
            term   = raw_term.lower()
            folder = term_dir(term)

            print(f"\n{'='*60}")
            print(f"[{i}/{total}]  {term}")
            print(f"{'='*60}")

            try:                                     # ← inner try is inside the for loop
                print("  Step 1: downloading daily chunks …")
                failed_chunks = download_daily_chunks(term, folder)

                print("  Step 2: downloading monthly anchor …")
                monthly_ok = download_monthly_anchor(term, folder)

                if failed_chunks:
                    msg = (f"Skipping stitch — {len(failed_chunks)} chunk(s) missing:\n" +
                           "\n".join(f"  {c}" for c in failed_chunks))
                    print(f"  WARNING: {msg}")
                    (folder / "incomplete.txt").write_text(
                        f"term: {term}\n"
                        f"status: incomplete\n"
                        f"reason: {len(failed_chunks)} chunk(s) failed to download\n\n"
                        f"failed chunks:\n" +
                        "\n".join(failed_chunks),
                        encoding="utf-8"
                    )
                    errors.append((term, f"{len(failed_chunks)} chunk(s) failed"))
                    time.sleep(30)
                    continue                          # ← now correctly inside for loop

                print("  Step 3: classifying files …")
                daily_files, monthly_file = classify_files(folder)

                if not daily_files:
                    raise ValueError("No daily files found after download.")

                print("  Step 4: stitching daily chunks …")
                stitched = stitch_daily(daily_files)

                if monthly_ok and monthly_file:
                    print("  Step 5: applying local monthly anchor …")
                    stitched = apply_monthly_anchor_local(stitched, monthly_file)
                else:
                    print("  WARNING: no monthly anchor — skipping calibration.")

                print("  Step 6: saving final series …")
                final = save_final(stitched, term, folder)

                if monthly_ok and monthly_file:
                    print("  Saving diagnostic txt …")
                    save_diagnostic(final, monthly_file, term, folder)

                incomplete = folder / "incomplete.txt"
                if incomplete.exists():
                    incomplete.unlink()

            except KeyboardInterrupt:
                raise                                # ← bubble up to outer handler

            except Exception as e:
                print(f"  ERROR on term '{term}': {e}")
                errors.append((term, str(e)))

            time.sleep(random.uniform(20, 45))       # inter-term cooldown

    except KeyboardInterrupt:
        print(f"\n  Last attempted term : {term}")
        print(f"  Set START_FROM = '{term}' to resume from here.")

    finally:
        print(f"\n{'='*60}")
        print(f"Pipeline stopped.")

Resuming from 'vagrant' (5 terms remaining)

[146/150]  vagrant
  Step 1: downloading daily chunks …
    [skip] 2022-01-01 → 2022-09-26 (already on disk)
    [skip] 2022-08-29 → 2023-05-24 (already on disk)
    [skip] 2023-04-26 → 2024-01-19 (already on disk)
    [skip] 2023-12-22 → 2024-09-15 (already on disk)
    [skip] 2024-08-18 → 2025-05-13 (already on disk)
    [skip] 2025-04-15 → 2026-01-08 (already on disk)
    [skip] 2025-12-11 → 2026-05-31 (already on disk)
  Step 2: downloading monthly anchor …
    [skip] monthly anchor (already on disk)
  Step 3: classifying files …
  Step 4: stitching daily chunks …
  Step 5: applying local monthly anchor …
  Step 6: saving final series …
    Saved: gt_stitched_2022-01-01_2026-05-31.csv  (1612 rows)
  Saving diagnostic txt …
    Diagnostic saved: diagnostic.txt  (0 bad months)

[147/150]  valuable
  Step 1: downloading daily chunks …
    [skip] 2022-01-01 → 2022-09-26 (already on disk)
    [skip] 2022-08-29 → 2023-05-24 (already on disk)
 

In [14]:
# Diagnosis of missiing chunks
from pathlib import Path

folders = [f for f in BASE_DIR.iterdir() if f.is_dir()]

print("Number of word folders:", len(folders))

for folder in folders[:5]:
    files = list(folder.glob("gt_daily_*.csv"))
    print(folder.name, len(files))

Number of word folders: 153
abundance 7
accrue 7
advantage 7
affluence 7
affluent 7


In [15]:
import re
from datetime import datetime

def extract_dates(filename):
    m = re.search(
        r"gt_daily_(\d{4}-\d{2}-\d{2})_(\d{4}-\d{2}-\d{2})",
        filename
    )
    return (
        datetime.strptime(m.group(1), "%Y-%m-%d"),
        datetime.strptime(m.group(2), "%Y-%m-%d")
    )


folder = folders[0]

chunks = sorted(folder.glob("gt_daily_*.csv"))

for f in chunks[:5]:
    print(f.name, extract_dates(f.name))

gt_daily_2022-01-01_2022-09-26.csv (datetime.datetime(2022, 1, 1, 0, 0), datetime.datetime(2022, 9, 26, 0, 0))
gt_daily_2022-08-29_2023-05-24.csv (datetime.datetime(2022, 8, 29, 0, 0), datetime.datetime(2023, 5, 24, 0, 0))
gt_daily_2023-04-26_2024-01-19.csv (datetime.datetime(2023, 4, 26, 0, 0), datetime.datetime(2024, 1, 19, 0, 0))
gt_daily_2023-12-22_2024-09-15.csv (datetime.datetime(2023, 12, 22, 0, 0), datetime.datetime(2024, 9, 15, 0, 0))
gt_daily_2024-08-18_2025-05-13.csv (datetime.datetime(2024, 8, 18, 0, 0), datetime.datetime(2025, 5, 13, 0, 0))


In [17]:
from pathlib import Path
import pandas as pd


data_dir = Path(r"C:\Python\CSUREMM\test")

folders = [f for f in data_dir.iterdir() if f.is_dir()]


missing_report = []


def inspect_word(folder):

    chunk_files = list(folder.glob("gt_daily_*.csv"))
    stitched_file = folder / "gt_stitched_2022-01-01_2026-05-31.csv"

    result = {
        "word": folder.name,
        "chunk_count": len(chunk_files),
        "stitched_exists": stitched_file.exists(),
        "issue": ""
    }

    # check missing stitched file
    if not stitched_file.exists():
        result["issue"] = "missing stitched file"
        return result

    # check empty stitched file
    try:
        stitched = pd.read_csv(stitched_file)

        if len(stitched) == 0:
            result["issue"] = "empty stitched file"

    except Exception as e:
        result["issue"] = f"read error: {e}"


    # check chunks
    if len(chunk_files) != 270:
        result["issue"] += f" | chunk count={len(chunk_files)}"


    return result



results = []

for folder in folders:
    results.append(inspect_word(folder))


diagnosis = pd.DataFrame(results)


# save full diagnosis
diagnosis.to_csv(
    "gt_stitch_diagnosis.csv",
    index=False
)


# create txt list only problematic words
problems = diagnosis[
    diagnosis["issue"] != ""
]


with open("missing_time_series.txt", "w", encoding="utf-8") as f:
    for _, row in problems.iterrows():
        f.write(
            f"{row['word']}: {row['issue']}\n"
        )


print("Total folders:", len(folders))
print("Problem folders:", len(problems))
print(problems.head(20))

Total folders: 153
Problem folders: 153
            word  chunk_count  stitched_exists             issue
0      abundance            7             True   | chunk count=7
1         accrue            7             True   | chunk count=7
2      advantage            7             True   | chunk count=7
3      affluence            7             True   | chunk count=7
4       affluent            7             True   | chunk count=7
5         afloat            7             True   | chunk count=7
6      allowance            7             True   | chunk count=7
7    aristocracy            7             True   | chunk count=7
8     aristocrat            7             True   | chunk count=7
9   aristocratic            7             True   | chunk count=7
10     associate            7             True   | chunk count=7
11        backer            7             True   | chunk count=7
12      backward            7             True   | chunk count=7
13  backwardness            7             True   |

In [ ]:
from pathlib import Path

data_dir = Path(r"C:\Python\CSUREMM\data_primitive")

folders = [
    f for f in data_dir.iterdir()
    if f.is_dir()
]

print("Current folders:", len(folders))

for f in folders[:10]:
    print(f)

Current folders: 150
C:\Python\CSUREMM\test\abundance
C:\Python\CSUREMM\test\accrue
C:\Python\CSUREMM\test\advantage
C:\Python\CSUREMM\test\affluence
C:\Python\CSUREMM\test\affluent
C:\Python\CSUREMM\test\afloat
C:\Python\CSUREMM\test\allowance
C:\Python\CSUREMM\test\aristocracy
C:\Python\CSUREMM\test\aristocrat
C:\Python\CSUREMM\test\aristocratic


In [55]:
from pathlib import Path
import pandas as pd


def fix_stitch(folder: Path, weekly_file: Path):

    # --------------------------------------------------
    # Step 1: rebuild daily series from all chunks
    # --------------------------------------------------

    chunks = sorted(
        folder.glob("gt_daily_*.csv")
    )

    dfs = []

    for f in chunks:
        df = pd.read_csv(
            f,
            parse_dates=["Time"]
        )
        dfs.append(df)

    stitched = pd.concat(
        dfs,
        ignore_index=True
    )

    value_col = [
        c for c in stitched.columns
        if c != "Time"
    ][0]

    stitched = (
        stitched
        .sort_values(
            ["Time", value_col],
            na_position="last"
        )
        .drop_duplicates(
            subset="Time",
            keep="first"
        )
        .sort_values("Time")
        .reset_index(drop=True)
    )

    # --------------------------------------------------
    # Step 2: load weekly anchor
    # --------------------------------------------------

    weekly = (
        pd.read_csv(
            weekly_file,
            parse_dates=["Time"]
        )
        .set_index("Time")
        .squeeze()
    )

    weekly.name = "weekly"

    # --------------------------------------------------
    # Step 3: compute weekly means from stitched series
    # --------------------------------------------------

    daily_agg = (
        stitched
        .set_index("Time")[value_col]
        .resample("W-SUN")
        .mean()
    )

    daily_agg.name = "daily_agg"

    comparison = (
        pd.concat(
            [weekly, daily_agg],
            axis=1
        )
        .dropna()
    )

    comparison = comparison[
        comparison["daily_agg"] > 1e-9
    ]

    comparison["scale"] = (
        comparison["weekly"]
        / comparison["daily_agg"]
    )

    # --------------------------------------------------
    # Step 4: assign ONE scale per week
    # --------------------------------------------------

    result = (
        stitched.copy()
        .set_index("Time")
    )

    week_end = (
        result.index
        .to_period("W-SUN")
        .end_time
        .normalize()
    )

    result["scale"] = pd.Series(
        week_end,
        index=result.index
    ).map(
        comparison["scale"]
    )

    result["scale"] = (
        result["scale"]
        .ffill()
        .bfill()
    )

    # --------------------------------------------------
    # Step 5: scale daily values
    # --------------------------------------------------

    result[value_col] = (
        result[value_col]
        * result["scale"]
    )

    result = result.drop(
        columns="scale"
    )

    return result.reset_index()

In [ ]:
folder = Path(
    r"C:\Python\CSUREMM\dictionary V0.0\data_primitive\abundance"
)

weekly_file = (
    folder /
    "gt_monthly_2022-01-01_2026-05-31.csv"
)


fixed = fix_stitch(
    folder,
    weekly_file
)


print(fixed.head())
print(fixed["value"].isna().sum())

        Time      value
0 2022-01-01  66.257669
1 2022-01-02  41.742331
2 2022-01-03  54.254743
3 2022-01-04  69.051491
4 2022-01-05  53.268293
0


In [58]:
def new_diagnostic(
    fixed: pd.DataFrame,
    weekly_file,
    term: str,
    folder: Path
):
    """
    Google Trends weekly-anchor diagnostic.

    Measures:
    - daily series integrity
    - weekly anchor alignment
    - ratio stability
    - abnormal weeks under multiple thresholds

    Does NOT discard data.
    """

    value_col = [
        c for c in fixed.columns
        if c != "Time"
    ][0]


    # -----------------------------
    # Load weekly anchor
    # -----------------------------
    
    weekly = (
        pd.read_csv(
            weekly_file,
            parse_dates=["Time"]
        )
        .set_index("Time")
        .squeeze()
    )

    weekly.name = "anchor"


    # -----------------------------
    # Aggregate fixed daily to weekly
    # -----------------------------

    fixed_weekly = (
        fixed
        .set_index("Time")[value_col]
        .resample("W-SUN")
        .mean()
    )

    fixed_weekly.name = "fixed"


    comparison = (
        pd.concat(
            [weekly, fixed_weekly],
            axis=1
        )
        .dropna()
    )


    comparison["ratio"] = (
        comparison["fixed"] /
        comparison["anchor"].replace(0, pd.NA)
    )


    comparison["deviation"] = (
        comparison["ratio"] - 1
    ).abs()


    # thresholds
    mild = (
        (comparison["ratio"] < 0.85) |
        (comparison["ratio"] > 1.15)
    )

    moderate = (
        (comparison["ratio"] < 0.75) |
        (comparison["ratio"] > 1.25)
    )

    severe = (
        (comparison["ratio"] < 0.50) |
        (comparison["ratio"] > 2.00)
    )


    # -----------------------------
    # Series integrity
    # -----------------------------

    missing = fixed[value_col].isna()


    # -----------------------------
    # Report
    # -----------------------------

    lines = [

        "Google Trends Weekly Anchor Diagnostic",
        "=" * 55,
        f"Term: {term.lower()}",
        f"Date range: {fixed['Time'].min().date()} → {fixed['Time'].max().date()}",
        "",

        "Daily series integrity:",
        f"  Daily observations       : {len(fixed)}",
        f"  Missing daily values     : {missing.sum()}",
        f"  Minimum value            : {fixed[value_col].min():.3f}",
        f"  Maximum value            : {fixed[value_col].max():.3f}",
        "",

        "Weekly anchor alignment:",
        f"  Anchor weeks compared    : {len(comparison)}",
        f"  Median ratio             : {comparison['ratio'].median():.3f}",
        f"  Median abs deviation     : {comparison['deviation'].median():.3f}",
        f"  Maximum deviation        : {comparison['deviation'].max():.3f}",
        "",

        "Quality flags:",
        f"  Mild abnormal weeks (0.85-1.15): "
        f"{mild.sum()} ({mild.mean()*100:.1f}%)",

        f"  Moderate abnormal weeks (0.75-1.25): "
        f"{moderate.sum()} ({moderate.mean()*100:.1f}%)",

        f"  Severe abnormal weeks (0.50-2.00): "
        f"{severe.sum()} ({severe.mean()*100:.1f}%)",

    ]


    # -----------------------------
    # Abnormal week table
    # -----------------------------

    bad = comparison[mild]


    if bad.empty:

        lines.append(
            "No weeks outside mild threshold."
        )

    else:

        lines.extend([
            "Weeks outside mild threshold:",
            "",
            f"{'Time':<15} {'anchor':>10} {'fixed':>10} {'ratio':>8}",
            "-" * 50
        ])

        for idx, row in bad.iterrows():

            lines.append(
                f"{str(idx.date()):<15} "
                f"{row['anchor']:>10.2f} "
                f"{row['fixed']:>10.2f} "
                f"{row['ratio']:>8.3f}"
            )


    # -----------------------------
    # Save
    # -----------------------------

    txt_path = folder / "new_diagnostic.txt"

    txt_path.write_text(
        "\n".join(lines),
        encoding="utf-8"
    )


    print(
        f"Diagnostic saved: {txt_path.name} | "
        f"mild={mild.sum()}, "
        f"moderate={moderate.sum()}, "
        f"severe={severe.sum()}"
    )

In [ ]:
data_dir = Path(
    r"C:\Python\CSUREMM\dictionary V0.0\data_primitive"
)


failed = []


for folder in data_dir.iterdir():

    if not folder.is_dir():
        continue


    try:

        weekly_file = (
            folder /
            "gt_monthly_2022-01-01_2026-05-31.csv"
        )


        # rebuild + anchor fix
        fixed = fix_stitch(
            folder,
            weekly_file
        )


        # save fixed series
        output = (
            Path("C:/Python/CSUREMM/dictionary V0.0/data_primitive_for_events") /
            f"gt_fixed_{folder.name}.csv"
        )

        fixed.to_csv(
            output,
            index=False
        )

        print("done:", folder.name)


    except Exception as e:

        failed.append(
            (folder.name, str(e))
        )

Diagnostic saved: new_diagnostic.txt | mild=0, moderate=0, severe=0
done: abundance
Diagnostic saved: new_diagnostic.txt | mild=0, moderate=0, severe=0
done: accrue
Diagnostic saved: new_diagnostic.txt | mild=0, moderate=0, severe=0
done: advantage
Diagnostic saved: new_diagnostic.txt | mild=16, moderate=16, severe=16
done: affluence
Diagnostic saved: new_diagnostic.txt | mild=0, moderate=0, severe=0
done: affluent
Diagnostic saved: new_diagnostic.txt | mild=0, moderate=0, severe=0
done: afloat
Diagnostic saved: new_diagnostic.txt | mild=0, moderate=0, severe=0
done: allowance
Diagnostic saved: new_diagnostic.txt | mild=0, moderate=0, severe=0
done: aristocracy
Diagnostic saved: new_diagnostic.txt | mild=0, moderate=0, severe=0
done: aristocrat
Diagnostic saved: new_diagnostic.txt | mild=1, moderate=1, severe=1
done: aristocratic
Diagnostic saved: new_diagnostic.txt | mild=0, moderate=0, severe=0
done: associate
Diagnostic saved: new_diagnostic.txt | mild=0, moderate=0, severe=0
done: 

In [ ]:
from pathlib import Path
import pandas as pd
import re

ROOT = Path(r"C:\Python\CSUREMM\data_primitive")

rows = []

for folder in ROOT.iterdir():

    if not folder.is_dir():
        continue

    txt_file = folder / "new_diagnostic.txt"

    if not txt_file.exists():
        continue

    text = txt_file.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    row = {
        "term": folder.name
    }

    patterns = {
        "daily_obs":
            r"Daily observations\s*:\s*(\d+)",

        "missing_daily":
            r"Missing daily values\s*:\s*(\d+)",

        "min_value":
            r"Minimum value\s*:\s*([0-9.]+)",

        "max_value":
            r"Maximum value\s*:\s*([0-9.]+)",

        "anchor_weeks":
            r"Anchor weeks compared\s*:\s*(\d+)",

        "median_ratio":
            r"Median ratio\s*:\s*([0-9.]+)",

        "median_abs_dev":
            r"Median abs deviation\s*:\s*([0-9.]+)",

        "max_deviation":
            r"Maximum deviation\s*:\s*([0-9.]+)",

        "mild_weeks":
            r"Mild abnormal weeks.*?:\s*(\d+)",

        "moderate_weeks":
            r"Moderate abnormal weeks.*?:\s*(\d+)",

        "severe_weeks":
            r"Severe abnormal weeks.*?:\s*(\d+)"
    }

    for col, pattern in patterns.items():

        match = re.search(pattern, text)

        if match:
            value = match.group(1)

            try:
                row[col] = float(value)
            except:
                row[col] = value

        else:
            row[col] = None

    rows.append(row)

diag_df = pd.DataFrame(rows)

In [62]:
diag_df = (
    diag_df
    .sort_values(
        ["severe_weeks",
         "moderate_weeks",
         "mild_weeks"],
        ascending=False
    )
)

print(diag_df.head(20))

             term  daily_obs  missing_daily  min_value  max_value  \
140  unprofitable     1612.0            0.0      0.000    609.000   
58      economize     1612.0            0.0      0.000    700.000   
24      betrothal     1612.0            0.0      0.000    581.000   
13   backwardness     1612.0            0.0      0.000    406.000   
128      squander     1612.0            0.0      0.000    630.000   
59          endow     1612.0            0.0      0.000     42.000   
3       affluence     1612.0            0.0      0.000    315.000   
23       bequeath     1612.0            0.0      0.000    581.000   
121      richness     1612.0            0.0      0.000    177.231   
130     subsidize     1612.0            0.0      0.000    303.614   
9    aristocratic     1612.0            0.0      0.000    171.779   
49         debtor     1612.0            0.0      0.000    133.333   
85      liquidate     1612.0            0.0      0.000    236.353   
89    meritorious     1612.0      

In [63]:
diag_df.to_csv(
    ROOT / "diagnostic_summary.csv",
    index=False
)

In [64]:
bad = diag_df.query(
    "median_ratio < 0.5"
)

print(
    bad[
        [
            "term",
            "median_ratio",
            "mild_weeks",
            "max_value"
        ]
    ].head(20)
)

             term  median_ratio  mild_weeks  max_value
140  unprofitable           0.0       132.0      609.0
58      economize           0.0       126.0      700.0
13   backwardness           0.0        42.0      406.0
